# Celabot — Paso 2: detección real con Supervision

En `hello.ipynb` decíamos "si hay personas → llama al VLM". Tonto pero útil para entender el patrón. Aquí lo reemplazamos por **tres reglas reales**:

| Caso | Cómo lo detectamos |
|---|---|
| **Intrusión fuera de horario** | Hay una persona dentro de la zona de la tienda en horario cerrado |
| **Merodeo** | El mismo `track_id` lleva `>N` segundos dentro de una zona vigilada |
| **Cruce de salida** | Un `track_id` cruza la línea de salida (futuro: sin POS previo = `grab-and-run`) |

Para esto agregamos tres primitivas nuevas:

- **Tracking**: asignamos un ID estable a cada persona entre frames (`sv.ByteTrack`). Sin tracking no podemos hablar de "esta persona específica", solo de "hay N personas".
- **Zonas**: polígonos en coordenadas de píxel; preguntamos a Supervision "¿esta detección está dentro?" (`sv.PolygonZone`).
- **Líneas**: segmentos que detectan cruces en cualquier dirección (`sv.LineZone`).

Todo esto sigue siendo **carril rápido** (CPU/GPU local, gratis). El VLM solo entra cuando una de las tres reglas dispara.

## 0. Setup

Mismas dependencias que `hello.ipynb` más `supervision`.

In [ ]:
!pip install -q ultralytics supervision google-genai opencv-python-headless

## 1. Qué pone Supervision sobre la mesa

Supervision no entrena ni hace inferencia. Es la capa de **post-proceso** entre el detector y tu lógica:

```
  YOLO  ──►  sv.Detections.from_ultralytics(...)  ──►  ByteTrack  ──►  Zonas/Líneas  ──►  Anotadores
                                                                                          │
                                                                                          ▼
                                                                                       video anotado
                                                                                       + estadísticas
```

Lo que vamos a usar concretamente:

- `sv.Detections.from_ultralytics(results)` — convierte la salida de YOLO al objeto estándar de Supervision.
- `sv.ByteTrack` — asigna `tracker_id` a cada detección con un algoritmo de tracking.
- `sv.PolygonZone(polygon)` — `.trigger(detections)` devuelve una máscara booleana "¿dentro?".
- `sv.LineZone(start, end)` — `.trigger(detections)` devuelve dos máscaras: cruces hacia adentro y hacia afuera.
- `sv.BoxAnnotator`, `sv.LabelAnnotator`, `sv.TraceAnnotator` — dibujan cajas, etiquetas y trayectorias sobre el frame.
- `sv.process_video` — abre un archivo, llama a tu función por cada frame y escribe el resultado.

Eso es todo el vocabulario que necesitas.

## 2. Sube tu video

Usa uno donde se vea movimiento de gente. Idealmente 20–60 s y resolución ≥ 480p.

In [ ]:
from google.colab import files
import cv2

uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]

cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f'Video: {VIDEO_PATH}')
print(f'Resolución: {W}x{H}  |  fps: {FPS:.1f}  |  duración: {TOTAL_FRAMES/FPS:.1f}s')

## 3. Mira el primer frame

Necesitamos definir **dónde** está la zona de la tienda y **dónde** está la línea de salida. Para no obligarte a editar coordenadas, los valores por defecto cubren el frame entero y ponen la línea cerca del borde inferior — útiles para sentir el sistema. Cuando quieras refinar, vuelve a esta celda, lee píxeles con el cursor sobre la imagen y ajusta el config en la siguiente.

In [ ]:
import matplotlib.pyplot as plt
import cv2

cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
cap.release()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
plt.grid(True, alpha=0.3)
plt.title(f'Primer frame ({W}x{H}). Lee coordenadas en píxeles si quieres ajustar zona/línea.')
plt.show()

## 4. Configura zona, línea y reglas

Coordenadas **normalizadas** (0.0 a 1.0): el sistema las multiplica por `W` y `H` así no dependes de la resolución del video.

- `ZONE_NORM`: polígono. Por defecto, casi todo el frame (5% de margen).
- `EXIT_LINE_NORM`: una línea horizontal al 80% de altura.
- `LOITERING_THRESHOLD_S`: cuántos segundos del mismo `track_id` dentro de la zona cuentan como merodeo.
- `SIMULATE_OUT_OF_HOURS`: en el prototipo no tenemos reloj de tienda, así que asumimos que todo el video ocurre fuera de horario para activar la regla de intrusión. Cambia a `False` si solo te interesan merodeo y cruce de salida.

In [ ]:
import numpy as np

ZONE_NORM = np.array([
    [0.05, 0.05],
    [0.95, 0.05],
    [0.95, 0.95],
    [0.05, 0.95],
])

EXIT_LINE_NORM = ((0.05, 0.80), (0.95, 0.80))

LOITERING_THRESHOLD_S = 5.0
SIMULATE_OUT_OF_HOURS = True

# Convertir a píxeles
ZONE_PX = (ZONE_NORM * np.array([W, H])).astype(np.int32)
EXIT_START_PX = (int(EXIT_LINE_NORM[0][0] * W), int(EXIT_LINE_NORM[0][1] * H))
EXIT_END_PX   = (int(EXIT_LINE_NORM[1][0] * W), int(EXIT_LINE_NORM[1][1] * H))

print(f'Zona en píxeles: {ZONE_PX.tolist()}')
print(f'Línea de salida en píxeles: {EXIT_START_PX} -> {EXIT_END_PX}')

## 5. Helpers reutilizables

Las mismas dos funciones de `hello.ipynb`, ahora limpias y como funciones de primera clase. `extract_clip` recorta un trozo del video original con ffmpeg, `ask_gemini` sube el clip y devuelve el veredicto JSON.

In [ ]:
import os, subprocess, time, json

def extract_clip(src: str, start: float, end: float, out: str) -> str:
    if os.path.exists(out):
        os.remove(out)
    duration = max(0.5, end - start)
    cmd = [
        'ffmpeg', '-y', '-loglevel', 'error',
        '-i', src,
        '-ss', f'{start:.2f}',
        '-t', f'{duration:.2f}',
        '-c:v', 'libx264', '-preset', 'ultrafast',
        '-movflags', '+faststart',
        '-an',
        out,
    ]
    subprocess.run(cmd, check=True)
    return out

def ask_gemini(client, clip_path: str, prompt: str, model: str = 'gemini-2.5-flash') -> dict:
    video_file = client.files.upload(file=clip_path)
    while video_file.state.name == 'PROCESSING':
        time.sleep(2)
        video_file = client.files.get(name=video_file.name)
    response = client.models.generate_content(
        model=model,
        contents=[video_file, prompt],
    )
    raw = response.text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.strip()
    return json.loads(raw)

## 6. Carril rápido con tracking + zona + línea

Ahora viene el corazón del paso 2. Por cada frame:

1. YOLO detecta personas.
2. `ByteTrack` les asigna un `tracker_id` estable.
3. `PolygonZone` nos dice cuáles están dentro de la zona.
4. `LineZone` nos dice cuáles cruzaron la línea de salida en este frame.
5. Acumulamos dos estructuras: `track_zone_times` (cuándo entró/salió cada track de la zona) y `exit_crossings` (lista de cruces).
6. Dibujamos cajas, IDs, trayectoria, polígono y línea sobre el frame.

Procesamos **todos los frames** (no muestreamos): el tracker necesita continuidad para no confundir personas distintas. Para videos de prototipo (≤ 60 s) está bien; en producción se hace en GPU con batches.

In [ ]:
from ultralytics import YOLO
import supervision as sv
import numpy as np
import cv2

model = YOLO('yolo11n.pt')
tracker = sv.ByteTrack()
zone = sv.PolygonZone(polygon=ZONE_PX)
line_zone = sv.LineZone(start=sv.Point(*EXIT_START_PX), end=sv.Point(*EXIT_END_PX))

box_ann = sv.BoxAnnotator()
label_ann = sv.LabelAnnotator()
trace_ann = sv.TraceAnnotator()

# Acumuladores cross-frame
track_zone_times: dict[int, dict[str, float]] = {}
exit_crossings: list[dict] = []

def callback(frame: np.ndarray, frame_idx: int) -> np.ndarray:
    ts = frame_idx / FPS

    # 1. Detección
    results = model(frame, verbose=False, classes=[0])[0]  # 0 = person
    detections = sv.Detections.from_ultralytics(results)

    # 2. Tracking
    detections = tracker.update_with_detections(detections)

    # 3. Zona
    if len(detections) > 0:
        in_zone = zone.trigger(detections)
        if detections.tracker_id is not None:
            for i in np.where(in_zone)[0]:
                tid = detections.tracker_id[i]
                if tid is None:
                    continue
                rec = track_zone_times.setdefault(int(tid), {'first': ts, 'last': ts})
                rec['last'] = ts

        # 4. Línea
        crossed_in, crossed_out = line_zone.trigger(detections)
        if detections.tracker_id is not None:
            for i in np.where(crossed_in)[0]:
                tid = detections.tracker_id[i]
                if tid is not None:
                    exit_crossings.append({'tid': int(tid), 'ts': ts, 'dir': 'in'})
            for i in np.where(crossed_out)[0]:
                tid = detections.tracker_id[i]
                if tid is not None:
                    exit_crossings.append({'tid': int(tid), 'ts': ts, 'dir': 'out'})

    # 5. Anotación
    annotated = frame.copy()
    if len(detections) > 0:
        annotated = trace_ann.annotate(annotated, detections)
        annotated = box_ann.annotate(annotated, detections)
        if detections.tracker_id is not None:
            labels = [f'#{int(t)}' for t in detections.tracker_id]
            annotated = label_ann.annotate(annotated, detections, labels)
    # Zona y línea dibujadas con cv2 (independiente de la versión de supervision)
    cv2.polylines(annotated, [ZONE_PX], isClosed=True, color=(0, 0, 255), thickness=2)
    cv2.line(annotated, EXIT_START_PX, EXIT_END_PX, (0, 255, 255), 2)
    return annotated

print('Procesando video con tracking + zona + línea (puede tomar 1-3 min sin GPU)...')
sv.process_video(source_path=VIDEO_PATH, target_path='annotated_raw.mp4', callback=callback)
print('Listo.')
print(f'Tracks distintos vistos en la zona: {len(track_zone_times)}')
print(f'Cruces de línea totales: {len(exit_crossings)}')

## 7. Mira el video anotado

Re-codificamos a un MP4 web-friendly y lo embebemos. Vas a ver las cajas con su `#tid`, la trayectoria reciente de cada persona, el polígono rojo (zona) y la línea amarilla (salida).

In [ ]:
import subprocess
from IPython.display import HTML
from base64 import b64encode

subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error',
    '-i', 'annotated_raw.mp4',
    '-c:v', 'libx264', '-preset', 'ultrafast', '-movflags', '+faststart', '-an',
    'annotated.mp4',
], check=True)

mp4 = open('annotated.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f'<video width=720 controls><source src="{data_url}" type="video/mp4"></video>')

## 8. De estadísticas a candidatos

Las dos estructuras que acumulamos ya tienen toda la info que necesitamos. Derivamos los candidatos para cada caso:

- **Intrusión**: si `SIMULATE_OUT_OF_HOURS`, cada track visto dentro de la zona es un candidato.
- **Merodeo**: tracks con duración `last - first >= LOITERING_THRESHOLD_S`.
- **Cruce de salida**: cada entrada en `exit_crossings`. Envolvemos en una ventana de ±3 s para darle contexto al VLM.

In [ ]:
intrusion_candidates = []
if SIMULATE_OUT_OF_HOURS:
    for tid, t in track_zone_times.items():
        intrusion_candidates.append({
            'tid': tid,
            'start': max(0, t['first'] - 1),
            'end': t['last'] + 1,
        })

loitering_candidates = []
for tid, t in track_zone_times.items():
    duration = t['last'] - t['first']
    if duration >= LOITERING_THRESHOLD_S:
        loitering_candidates.append({
            'tid': tid,
            'start': t['first'],
            'end': t['last'],
            'duration': duration,
        })

exit_candidates = []
for crossing in exit_crossings:
    exit_candidates.append({
        'tid': crossing['tid'],
        'ts': crossing['ts'],
        'dir': crossing['dir'],
        'start': max(0, crossing['ts'] - 3),
        'end': crossing['ts'] + 3,
    })

print(f'Candidatos de intrusión:   {len(intrusion_candidates)}')
print(f'Candidatos de merodeo:     {len(loitering_candidates)}')
print(f'Candidatos de cruce salida: {len(exit_candidates)}')

for c in loitering_candidates[:5]:
    print(f"  merodeo #{c['tid']}: {c['start']:.1f}s -> {c['end']:.1f}s ({c['duration']:.1f}s)")
for c in exit_candidates[:5]:
    print(f"  cruce #{c['tid']} dir={c['dir']} en t={c['ts']:.1f}s")

## 9. Carril lento: Gemini por caso

Cada caso necesita un *prompt* distinto. Lo importante: cada prompt es **estricto en hechos observables** y le decimos qué cuenta como normal vs anómalo en ese caso específico.

Para no quemar la cuota, en este notebook tomamos **un solo candidato por caso** (el primero). En producción mandas todos en paralelo con *rate limit* y cooldown por tienda.

In [ ]:
import getpass
from google import genai

GEMINI_API_KEY = getpass.getpass('Pega tu API key de Gemini: ')
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
PROMPT_INTRUSION = '''Eres un analista de seguridad. Esta tienda está cerrada (fuera de horario).
Mira el clip y responde ESTRICTAMENTE en JSON:
{"anomaly": bool, "confidence": float, "category": "intrusion" | "normal", "observed_behaviors": [str], "reason": str, "suggested_action": str}

Reglas:
- anomaly=true si la persona se comporta como intruso (forzar caja, abrir cajones, ocultar productos, no parecer empleado).
- anomaly=false si parece personal de limpieza, mantenimiento, o el dueño revisando.
- Solo el JSON, sin texto extra.'''

PROMPT_LOITERING = '''Eres un analista de seguridad. Esta persona estuvo {duration:.0f} segundos dentro de una zona vigilada (típicamente una góndola valiosa).
Responde ESTRICTAMENTE en JSON:
{{"anomaly": bool, "confidence": float, "category": "merodeo" | "normal", "observed_behaviors": [str], "reason": str, "suggested_action": str}}

Reglas:
- NORMAL: comparar productos, leer etiquetas, decidir, hablar por celular, esperar a alguien.
- SOSPECHOSO: vigilar empleados o cámaras, ocultar producto en ropa/bolso, mirar repetidamente alrededor antes de tomar algo.
- Solo el JSON.'''

PROMPT_EXIT = '''Eres un analista de seguridad. Una persona cruzó la línea cerca de la salida de la tienda.
Responde ESTRICTAMENTE en JSON:
{"anomaly": bool, "confidence": float, "category": "grab-and-run" | "salida_normal", "observed_behaviors": [str], "reason": str, "suggested_action": str}

Reglas:
- anomaly=true SOLO si ves: persona corriendo con productos sin embolsar/pagar, evasión visible de empleados, urgencia atípica.
- anomaly=false si la salida luce normal (camina, lleva bolsas, sin urgencia).
- Solo el JSON.'''

print('Prompts listos.')

In [ ]:
def run_case(name: str, candidate: dict, prompt: str, clip_name: str):
    print(f'\n=== {name.upper()} ===')
    print(f"Candidato: tid={candidate.get('tid')} ventana={candidate['start']:.1f}-{candidate['end']:.1f}s")
    end = min(candidate['end'], candidate['start'] + 10)
    extract_clip(VIDEO_PATH, candidate['start'], end, clip_name)
    try:
        verdict = ask_gemini(client, clip_name, prompt)
        print(json.dumps(verdict, indent=2, ensure_ascii=False))
    except Exception as e:
        print(f'Error con Gemini: {e}')

if intrusion_candidates:
    run_case('intrusión', intrusion_candidates[0], PROMPT_INTRUSION, 'clip_intrusion.mp4')
else:
    print('Sin candidatos de intrusión.')

if loitering_candidates:
    c = loitering_candidates[0]
    run_case('merodeo', c, PROMPT_LOITERING.format(duration=c['duration']), 'clip_loitering.mp4')
else:
    print('Sin candidatos de merodeo (ningún track pasó el umbral).')

if exit_candidates:
    run_case('cruce salida', exit_candidates[0], PROMPT_EXIT, 'clip_exit.mp4')
else:
    print('Sin cruces de la línea de salida.')

## 10. Qué cambió respecto al paso 1

| | Paso 1 (`hello.ipynb`) | Paso 2 (este) |
|---|---|---|
| Detección | "hay personas" | Personas con **ID estable** |
| Reglas | 1 (duración mínima) | 3 (intrusión, merodeo, cruce) |
| Espacial | Frame entero | **Zonas** y **líneas** configurables |
| Prompts | 1 genérico | 1 por caso, con reglas específicas |
| Falsos positivos | Altos | Mucho más bajos: el VLM solo se llama cuando una regla espacial dispara |

## Lo que sigue

Tres caminos posibles para el **paso 3**:

1. **Concealment con pose y manos** — añadir RTMPose para detectar la articulación de mano, seguir su trayectoria con un objeto en mano hasta el bolsillo/bolso. Esta es la detección estrella de hurto de cliente.
2. **RTSP en vivo** — cambiar `VIDEO_PATH` por un stream `rtsp://...` y simular ingesta en tiempo real con `mediaMTX` local. Aprendes el problema real de streaming.
3. **Detector de arma** — fine-tunear un YOLO con un dataset público (Sohas) y conectarlo a un prompt de Gemini específico para arma armada / armada en vitrina.

Dime cuál te llama más y arrancamos.